In [13]:
import sys
!"{sys.executable}" -m pip install xgboost

  Using cached xgboost-3.3.0-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.3.0-py3-none-win_amd64.whl (69.5 MB)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [15]:
data = np.load("../artifacts/dataset_sequences.npz")

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(216981, 24, 36)
(43963, 24, 36)
(28428, 24, 36)


In [16]:
pred = np.load("../artifacts/lstm_predictions.npz")

y_train_true = pred["y_train_true"]
y_val_true   = pred["y_val_true"]
y_test_true  = pred["y_test_true"]

y_train_pred = pred["y_train_pred"]
y_val_pred   = pred["y_val_pred"]
y_test_pred  = pred["y_test_pred"]

residual_train = pred["residual_train"]
residual_val   = pred["residual_val"]
residual_test  = pred["residual_test"]

In [34]:
# Ambil feature pada timestep terakhir dari setiap window
X_train_xgb = X_train[:, -1, :]
X_val_xgb   = X_val[:, -1, :]
X_test_xgb  = X_test[:, -1, :]

print("Shape sebelum tambah prediksi LSTM:")
print(X_train_xgb.shape)
print(X_val_xgb.shape)
print(X_test_xgb.shape)

Shape sebelum tambah prediksi LSTM:
(216981, 36)
(43963, 36)
(28428, 36)


In [35]:
# Tambahkan prediksi LSTM sebagai feature tambahan
X_train_xgb = np.hstack([
    X_train_xgb,
    y_train_pred.reshape(-1, 1)
])

X_val_xgb = np.hstack([
    X_val_xgb,
    y_val_pred.reshape(-1, 1)
])

X_test_xgb = np.hstack([
    X_test_xgb,
    y_test_pred.reshape(-1, 1)
])

print("Shape akhir:")
print(X_train_xgb.shape)
print(X_val_xgb.shape)
print(X_test_xgb.shape)

Shape akhir:
(216981, 37)
(43963, 37)
(28428, 37)


In [36]:
os.makedirs("../models", exist_ok=True)

In [37]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    residual_train
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [38]:
train_residual_pred = xgb_model.predict(X_train_xgb)
val_residual_pred = xgb_model.predict(X_val_xgb)
test_residual_pred = xgb_model.predict(X_test_xgb)

In [39]:
train_final = y_train_pred.ravel() + train_residual_pred
val_final = y_val_pred.ravel() + val_residual_pred
test_final = y_test_pred.ravel() + test_residual_pred

In [40]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    nrmse = rmse / (np.max(y_true) - np.min(y_true))
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "NRMSE": nrmse,
        "R2": r2
    }

In [41]:
train_metrics = evaluate(y_train_true, train_final)
val_metrics = evaluate(y_val_true, val_final)
test_metrics = evaluate(y_test_true, test_final)
results = pd.DataFrame(
    [train_metrics, val_metrics, test_metrics],
    index=["Train", "Validation", "Test"]
)
results

,MAE,MSE,RMSE,NRMSE,R2
Train,2.837082,31.673278,5.627902,0.005396,0.995736
Validation,2.604982,32.259837,5.679774,0.008619,0.994277
Test,2.431621,23.942780,4.893136,0.008350,0.987781


In [42]:
lstm_results = pd.read_csv(
    "../models/lstm_evaluation.csv",
    index_col=0
)

lstm_results

,MAE,MSE,RMSE,NRMSE,R²
Train,3.210650,50.317642,7.093493,0.006801,0.993226
Validation,2.725367,32.782827,5.725629,0.008688,0.994184
Test,2.504920,23.959700,4.894865,0.008353,0.987772


In [43]:
lstm_results = lstm_results.rename(columns={"R2": "R²"})
results = results.rename(columns={"R2": "R²"})

In [44]:
comparison = pd.DataFrame({
    "LSTM": lstm_results.loc["Test"],
    "LSTM + XGBoost": results.loc["Test"]
}).T

comparison

,MAE,MSE,RMSE,NRMSE,R²
LSTM,2.504920,23.95970,4.894865,0.008353,0.987772
LSTM + XGBoost,2.431621,23.94278,4.893136,0.008350,0.987781


In [45]:
mae_improvement = (
    (lstm_results.loc["Test", "MAE"] - results.loc["Test", "MAE"])
    / lstm_results.loc["Test", "MAE"]
) * 100

rmse_improvement = (
    (lstm_results.loc["Test", "RMSE"] - results.loc["Test", "RMSE"])
    / lstm_results.loc["Test", "RMSE"]
) * 100

r2_improvement = (
    (results.loc["Test", "R²"] - lstm_results.loc["Test", "R²"])
    / abs(lstm_results.loc["Test", "R²"])
) * 100

print(f"Improvement MAE  : {mae_improvement:.2f}%")
print(f"Improvement RMSE : {rmse_improvement:.2f}%")
print(f"Improvement R²   : {r2_improvement:.2f}%")

Improvement MAE  : 2.93%
Improvement RMSE : 0.04%
Improvement R²   : 0.00%
